# AlphaNet Cryptocurrency Data Preparation

This notebook prepares cryptocurrency 15-minute OHLCV data for AlphaNet training, adapting the original stock methodology to crypto markets.

## Data Overview
- **Data Source**: 334 cryptocurrency parquet files with 15-minute intervals
- **Time Range**: ~2021-2024
- **AlphaNet Format**: 9×30 feature matrices with standardized return targets
- **Key Challenge**: Fill time gaps and adapt stock methodology to 24/7 crypto markets

## 1. Environment Setup and Dependencies

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
import psutil
import time
from joblib import Parallel, delayed

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [2]:
stable_list = ['USDCUSDT', 'USUALUSDT', 'USUALUSDT', 'TUSDT']

## 2. Robust Data Loading Functions

In [3]:
def read_parquet_robust(file_path: str) -> pd.DataFrame:
    
    """
    Robust parquet file reader with multiple engine fallbacks
    
    Args:
        file_path (str): Path to the parquet file
        
    Returns:
        pd.DataFrame: Loaded dataframe or empty dataframe if failed
    """
    df = pd.read_parquet(file_path)
    if not df.empty:
        # Column mapping for actual parquet structure
        column_mapping = {
            'open_price': 'open',
            'high_price': 'high', 
            'low_price': 'low',
            'close_price': 'close',
            # timestamp and volume are already correctly named
        }

        # Apply column mapping
        df = df.rename(columns=column_mapping)

        # Required columns after mapping
        required_cols = ['timestamp', 'open', 'high', 'low', 'close', 'volume']

        # Convert timestamp to datetime
        if df['timestamp'].dtype == 'object':
            df['timestamp'] = pd.to_datetime(df['timestamp'])
        elif df['timestamp'].dtype in ['int64', 'int32']:
            # Handle millisecond timestamps
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')

        # Convert price columns to float
        price_cols = ['open', 'high', 'low', 'close']
        for col in price_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        # Convert volume columns to numeric (handle object types)
        volume_cols = ['volume', 'amount', 'buy_volume', 'buy_amount']
        for col in volume_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        # Convert count to int
        if 'count' in df.columns:
            df['count'] = pd.to_numeric(df['count'], errors='coerce').fillna(0).astype('int64')

        # Sort by timestamp
        df = df.sort_values('timestamp').reset_index(drop=True)
        df['vwap'] = (df['amount'] / df['volume']).replace([np.inf, -np.inf], np.nan).ffill()

        # Keep only required columns (but also preserve additional useful columns)
        # Keep buy_volume and buy_amount for potential enhanced feature engineering
        available_extra_cols = [col for col in ['amount', 'count', 'buy_volume', 'buy_amount', 'vwap'] if col in df.columns]
        final_cols = required_cols + available_extra_cols
        df = df[final_cols]
    return df


def load_crypto_data(data_dir: str = '../data/raw') -> Dict[str, pd.DataFrame]:
    """
    Load all cryptocurrency parquet files with robust error handling
    
    Returns:
        Dict[str, pd.DataFrame]: Dictionary mapping symbol to dataframe
    """
    print(f" Loading cryptocurrency data from {data_dir}...")
    
    if not os.path.exists(data_dir):
        print(f"Data directory not found: {data_dir}")
        return {}
    
    STABLE = [x + '.parquet' for x in stable_list]
    
    parquet_files = [f for f in os.listdir(data_dir) if f.endswith('.parquet') and f not in STABLE]
    
    parquet_files
    print(f"Found {len(parquet_files)} parquet files")
    
    crypto_data = {}
    successful_loads = 0
    failed_loads = 0
    failed_symbols = []
    
    for filename in tqdm(parquet_files, desc="Loading files"):
        symbol = filename.replace('.parquet', '')
        file_path = os.path.join(data_dir, filename)
        
        try:
            # Load and standardize data
            df = read_parquet_robust(file_path)
            df['symbol'] = symbol
            
            if not df.empty and len(df) >= 100:  # Minimum data requirement
                crypto_data[symbol] = df
                successful_loads += 1
            else:
                failed_loads += 1
                failed_symbols.append(symbol)
                if len(df) < 100:
                    print(f"{symbol}: Insufficient data ({len(df)} records)")
                    
        except Exception as e:
            failed_loads += 1
            failed_symbols.append(symbol)
            print(f"Failed to load {symbol}: {str(e)[:50]}...")
    
    print(f"\\n Successfully loaded: {successful_loads} files")
    print(f"Failed to load: {failed_loads} files")
    
    if failed_symbols:
        print(f"\\n Failed symbols: {', '.join(failed_symbols[:10])}{'...' if len(failed_symbols) > 10 else ''}")
        print(f"Recommendation: These {failed_loads} coins will be excluded from AlphaNet training")
        print(f"This is normal - represents {failed_loads/len(parquet_files)*100:.1f}% failure rate")
    
    return crypto_data

## 2. Load and Process Data

In [4]:
# Load all cryptocurrency data
crypto_data = load_crypto_data()

print(f"\nLoaded data for {len(crypto_data)} cryptocurrencies")

# Show sample data
if crypto_data:
    sample_symbol = list(crypto_data.keys())[0] 
    sample_df = crypto_data[sample_symbol]
    print(f"\nSample data from {sample_symbol}:")
    print(sample_df.head())
    print(f"\nData types:")
    print(sample_df.dtypes)

 Loading cryptocurrency data from ../data/raw...
Found 352 parquet files


Loading files:   7%|████▋                                                             | 25/352 [00:09<01:28,  3.69it/s]

AI16ZUSDT: Insufficient data (0 records)


Loading files:   8%|█████                                                             | 27/352 [00:09<01:05,  4.94it/s]

ALCHUSDT: Insufficient data (0 records)


...

VINEUSDT: Insufficient data (0 records)
VTHOUSDT: Insufficient data (0 records)
VVVUSDT: Insufficient data (0 records)


Loading files:  98%|███████████████████████████████████████████████████████████████▉ | 346/352 [02:01<00:03,  2.00it/s]

ZEREBROUSDT: Insufficient data (0 records)


Loading files: 100%|█████████████████████████████████████████████████████████████████| 352/352 [02:03<00:00,  2.85it/s]

\n Successfully loaded: 331 files
Failed to load: 21 files
\n Failed symbols: AI16ZUSDT, ALCHUSDT, ANIMEUSDT, ARCUSDT, AVAAIUSDT, BIOUSDT, COOKIEUSDT, DUSDT, GRIFFAINUSDT, MELANIAUSDT...
Recommendation: These 21 coins will be excluded from AlphaNet training
This is normal - represents 6.0% failure rate

Loaded data for 331 cryptocurrencies

Sample data from 1000000MOGUSDT:
            timestamp    open    high     low   close     volume  \
0 2024-11-07 12:30:00  2.0896  2.1361  2.0560  2.0652   876381.4   
1 2024-11-07 12:45:00  2.0659  2.0730  1.9797  1.9898  1108330.3   
2 2024-11-07 13:00:00  1.9915  2.0132  1.9521  2.0072  1208569.3   
3 2024-11-07 13:15:00  2.0073  2.0283  1.9631  1.9805  1287539.7   
4 2024-11-07 13:30:00  1.9795  1.9958  1.9158  1.9917  1581100.1   

         amount  count  buy_volume    buy_amount      vwap          symbol  
0  1.824179e+06   6864    414306.5  8.635601e+05  2.081490  1000000MOGUSDT  
1  2.232821e+06   8529    461204.8  9.287305e+05  2.014581  1

In [5]:
abnormal_ones = []
for k in crypto_data:
    temp = crypto_data[k]
    if temp[temp['count'] == 0].shape[0] > 10:
        print(k)
        abnormal_ones.append(k)
        
for k in (abnormal_ones):
    crypto_data.pop(k)

ICPUSDT
TLMUSDT


In [6]:
test_data = pd.concat([crypto_data[k] for k in crypto_data][:20])

In [1]:
def calc_factor(df: pd.DataFrame, n: int = 20):
    """
    一次性计算 8 大方向的单因子（示例版本）
    参数
    ----
    df : pd.DataFrame
        必须包含全部 12 个字段，索引按 timestamp 升序
    n  : int
        回看窗口，默认为 20 根 bar
    """
    df = df.sort_values('timestamp')

    # 1) 价格动量类：
    df['momentum'] = (df['close'] / df['close'].shift(n) - 1)

    # 2) 成交量动量类：n 日均量相对最新量比
    df['vma'] = df['volume'] / df['volume'].rolling(n).mean()

    # 3) 价量背离类：OBV 的单期变化
    df['obv_chg'] = np.sign(df['close'] - df['close'].shift()) * df['volume']

    # 4) 买卖失衡类：成交量买单占比
    df['bsi'] = (df['buy_volume'] - (df['volume'] - df['buy_volume'])) / df['volume']

    # 5) VWAP 偏离
    df['vwap_dev'] = (df['close'] - df['vwap']) / df['vwap']

    # 6) 波动率类：Garman-Klass 波动率（n 日）
    O = df['open'].values
    H = df['high'].values
    L = df['low'].values
    C = df['close'].values
    
    # 计算对数价格比
    log_hl = np.log(H / L)
    log_co = np.log(C / O)
    
    # 计算每日波动率贡献
    daily_var = 0.5 * (log_hl)**2 - (2 * np.log(2) - 1) * (log_co)**2
    
    # 避免负方差（理论上不应该出现）
    daily_var = np.maximum(daily_var, 0)
    
    # 计算平均日波动率
    df['gk_volatility'] = np.sqrt(np.nanmean(daily_var))

    # 7) 盘口集中度：平均单笔成交数
    df['avg_trade_size'] = df['volume'] / df['count']

    # 8) 成交额相关：成交额 / 成交额 n 日均值
    df['amt_ratio'] = df['amount'] / df['amount'].rolling(n).mean()
    
    # 9) MACD信号：指数移动平均线差值，捕捉趋势变化
    exp1 = df['close'].ewm(span=12, adjust=False).mean()  # 12期EMA
    exp2 = df['close'].ewm(span=26, adjust=False).mean()  # 26期EMA
    macd = exp1 - exp2
    df['macd_signal'] = (macd - macd.ewm(span=9, adjust=False).mean()) / df['close']

    # 10) 布林带位置：价格在布林带中的相对位置，-1到1之间
    bb_mean = df['close'].rolling(20).mean()
    bb_std = df['close'].rolling(20).std()
    df['bb_position'] = (df['close'] - bb_mean) / (2 * bb_std)

    # 11) 威廉指标：衡量超买超卖，-100到0之间
    highest_high = df['high'].rolling(14).max()
    lowest_low = df['low'].rolling(14).min()
    df['williams_r'] = -100 * (highest_high - df['close']) / (highest_high - lowest_low)

    # 12) 随机指标K值：价格在最近高低区间的相对位置，0-100
    df['stoch_k'] = 100 * (df['close'] - lowest_low) / (highest_high - lowest_low)

    # 13) ATR（平均真实波幅）：衡量市场波动性
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df['atr'] = true_range.rolling(14).mean() / df['close']
    
    typical_price = (df['high'] + df['low'] + df['close']) / 3
    # 14) 资金流量指数（MFI）：结合价格和成交量的动量指标，0-100
    money_flow = typical_price * df['volume']
    positive_flow = money_flow.where(typical_price > typical_price.shift(), 0)
    negative_flow = money_flow.where(typical_price < typical_price.shift(), 0)
    mf_ratio = positive_flow.rolling(14).sum() / negative_flow.rolling(14).sum()
    df['mfi'] = 100 - (100 / (1 + mf_ratio))

    # 15) Chaikin资金流量：衡量资金流入流出强度，-1到1之间
    clv = ((df['close'] - df['low']) - (df['high'] - df['close'])) / (df['high'] - df['low'])
    df['cmf'] = (clv * df['volume']).rolling(20).sum() / df['volume'].rolling(20).sum()

    # 16) 商品通道指数（CCI）：衡量价格偏离统计平均值的程度
    cci_constant = 0.015
    mean_dev = (typical_price - typical_price.rolling(20).mean()).abs().rolling(20).mean()
    df['cci'] = (typical_price - typical_price.rolling(20).mean()) / (cci_constant *
    mean_dev)

    # 16) 上影线比率：上影线占K线实体的比例，反映上方压力
    df['upper_shadow'] = (df['high'] - df[['open', 'close']].max(axis=1)) / (df['high'] - df['low'] + 1e-10)

    # 17) 下影线比率：下影线占K线实体的比例，反映下方支撑
    df['lower_shadow'] = (df[['open', 'close']].min(axis=1) - df['low']) / (df['high'] - df['low'] + 1e-10)

    # 18) 实体比率：K线实体占整个波动范围的比例，反映趋势强度
    df['body_ratio'] = np.abs(df['close'] - df['open']) / (df['high'] - df['low'] + 1e-10)

    # 19) 价格加速度：价格变化的二阶导数，捕捉趋势加速或减速
    df['price_accel'] = df['close'].diff().diff() / df['close']

    # 20) 高低价差：波动幅度相对于收盘价的比例
    df['hl_spread'] = (df['high'] - df['low']) / df['close']

    # 21) 收盘价位置：收盘价在当日波动范围中的相对位置，0-1之间
    df['close_position'] = (df['close'] - df['low']) / (df['high'] - df['low'] + 1e-10)

    # 22) 价格效率：收盘价与VWAP的比率，衡量价格发现效率
    df['price_efficiency'] = df['close'] / df['vwap']

    # 23) K线内波动率：高低价差相对于开盘价的比例
    df['intrabar_vol'] = (df['high'] - df['low']) / df['open']
    
    return df


def add_feat(df: pd.DataFrame) -> tuple[pd.DataFrame, list]:
    """优化后的特征工程函数"""
    # 排序数据
    df = df.sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    
    # 计算目标变量 y_raw
    df["future_vwap"] = df.groupby("symbol")["vwap"].shift(-96)
    df["y_raw"] = df["future_vwap"] / df["vwap"] - 1
    
    # 计算特征
    df = df.groupby('symbol', group_keys=False).apply(calc_factor)
    
    # 特征列列表
    feature_cols = ['momentum', 'vma', 'obv_chg', 'bsi', 'vwap_dev', 'gk_volatility', 'macd_signal', 'bb_position', 'williams_r',
                    'stoch_k', 'atr', 'avg_trade_size', 'amt_ratio', 'cmf', 'upper_shadow', 'lower_shadow', 'body_ratio',
                    'price_accel', 'hl_spread', 'close_position', 'price_efficiency', 'intrabar_vol', 'mfi', 'cci']
    
    df = df.dropna()
    # =============== 优化后的去极值处理 ===============
    # 1. 向量化计算中位数和MAD
    grouped = df.groupby('timestamp')
    
    # 一次性计算所有特征的中位数
    medians = grouped[feature_cols].median().add_prefix('median_')
    
    # 一次性计算所有特征的MAD
    abs_devs = df[feature_cols].sub(medians.loc[df['timestamp']].values)
    abs_devs = abs_devs.abs()
    mads = grouped[abs_devs.columns].median().add_prefix('mad_')
    
    # 计算上下界
    k = 3.0
    scale_factor = 1.4826
    upper_bounds = pd.DataFrame(medians.values + (k * scale_factor * mads).values, index=medians.index, columns=feature_cols)
    lower_bounds = pd.DataFrame(medians.values - (k * scale_factor * mads).values, index=medians.index, columns=feature_cols)
    
    # 2. 向量化裁剪
    for col in feature_cols:
        col_vals = df[col].values
        upper = upper_bounds[col].loc[df['timestamp']].values
        lower = lower_bounds[col].loc[df['timestamp']].values
        
        # 使用numpy.clip进行向量化操作
        clipped = np.clip(col_vals, lower, upper)
        df[col] = clipped

    # 2. 标准化处理
    grouped = df.groupby('timestamp')
    min_ = grouped[feature_cols + ['y_raw']].min()
    max_ = grouped[feature_cols + ['y_raw']].max()

    # 标准化每个特征
    for col in feature_cols + ['y_raw']:
        # 获取当前时间戳的均值和标准差
        col_vals = df[col].values
        min_vals = min_[col].loc[df['timestamp']].values
        max_vals = max_[col].loc[df['timestamp']].values
        
        # 应用标准化公式: (x - mean) / std
        normalized = (col_vals - min_vals) / max_vals
        df[col] = normalized
    
    return df.reset_index(drop=True), feature_cols

NameError: name 'pd' is not defined

In [35]:
df_final, features = add_feat(test_data)
df_final.drop(['future_vwap'], axis=1, inplace=True)
df_final.to_parquet("panel.parquet")

In [36]:
df_final.isnull().sum(axis=0)

timestamp           0
open                0
high                0
low                 0
close               0
volume              0
amount              0
count               0
buy_volume          0
buy_amount          0
vwap                0
symbol              0
y_raw               0
momentum            0
vma                 0
obv_chg             0
bsi                 0
vwap_dev            0
gk_volatility       0
avg_trade_size      0
amt_ratio           0
macd_signal         0
bb_position         0
williams_r          0
stoch_k             0
atr                 0
mfi                 0
cmf                 0
cci                 0
upper_shadow        0
lower_shadow        0
body_ratio          0
price_accel         0
hl_spread           0
close_position      0
price_efficiency    0
intrabar_vol        0
dtype: int64

# 上面是改动后的数据准备，截面标准化只对features列做

# 可以尝试不用原始的数据，只用features输入到模型中预测

In [17]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
import torch.optim as optim
import math
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [18]:
class AlphaNetUniversalDataset(Dataset):
    def __init__(self, df, feat_columns, y_columns='y_raw', seq_len=96):
        self.seq_len = seq_len
        self.y_columns = y_columns

        # 存储每个symbol的x和y数据
        self.x_list = []  # 存储每个symbol的滑动窗口视图
        self.x_daily_list = []
        self.y_list = []  # 存储每个symbol的y值
        self.info_list = []

        # 存储每个symbol的样本数量
        self.sample_counts = []

        # 存储每个symbol的起始索引
        self.start_indices = []

        # 处理每个symbol的数据
        current_start = 0
        df = df.sort_values(['symbol', 'timestamp']).reset_index(drop=True)
        for symbol, group in df.groupby('symbol'): 
            if group.shape[0] < seq_len * 96:
                # 如果数据长度不足，跳过该symbol
                continue

            # 创建滑动窗口视图
            windowed_data = sliding_window_view(group[min_columns].values, window_shape=seq_len, axis=0).transpose(0, 2, 1)
            windowed_daily_data = sliding_window_view(group[daily_columns].values, window_shape=96 * seq_len, axis=0).transpose(0, 2, 1)
            start = windowed_data.shape[0] - windowed_daily_data.shape[0]
            windowed_data = windowed_data[start:]
            indices = [i * 96 for i in range(seq_len)]
            selected_view = windowed_daily_data[:, indices, :]
            # 存储数据
            self.x_list.append(windowed_data)
            self.y_list.append(group[y_columns].values[start:])
            self.info_list.append(group.index.values[start:])

            self.x_daily_list.append(selected_view)
            # 记录样本数量
            n_samples = windowed_data.shape[0]
            self.sample_counts.append(n_samples)

            # 记录起始索引
            self.start_indices.append(current_start)
            current_start += n_samples

        # 总样本数
        self.total_samples = current_start

        # 创建累积样本数数组，用于快速查找
        self.cumulative_counts = np.cumsum([0] + self.sample_counts)

    def _find_symbol_index(self, idx):
        """根据全局索引找到对应的symbol索引和局部索引"""
        # 使用二分查找找到对应的symbol索引
        symbol_idx = np.searchsorted(self.cumulative_counts, idx, side='right') - 1

        # 计算在该symbol中的局部索引
        local_idx = idx - self.cumulative_counts[symbol_idx]

        return symbol_idx, local_idx

    def __len__(self):
        return self.total_samples

    def __getitem__(self, idx):
        # 找到对应的symbol索引和局部索引
        symbol_idx, local_idx = self._find_symbol_index(idx)

        # 获取对应的数据
        x = self.x_list[symbol_idx][local_idx]
        x_daily = self.x_daily_list[symbol_idx][local_idx]
        x_full = np.concatenate([x, x_daily], axis=-1)
        y = self.y_list[symbol_idx][local_idx]
        info = self.info_list[symbol_idx][local_idx]

        # 转换为PyTorch张量
        x_tensor = torch.from_numpy(x_full.astype(np.float32))  # 1, len_seq: default 96, features 
        y_tensor = torch.tensor(y, dtype=torch.float32) # 1
        info_tensor = torch.tensor(info, dtype=torch.int64)  # 转换为整数张量
        return x_tensor, y_tensor, info_tensor

In [19]:
class CryptoGRU(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3, bidirectional=False):
        """
        GRU模型用于加密货币收益预测
        
        参数:
        - input_size: 输入特征维度 (n_features)
        - hidden_size: GRU隐藏层大小
        - num_layers: GRU层数
        - dropout: Dropout比例
        - bidirectional: 是否使用双向GRU
        """
        super(CryptoGRU, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        # GRU层
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        
        # 注意力机制
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * self.num_directions, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )
        
        # 输出层
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * self.num_directions, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1)
        )
        
        # 初始化权重
        self._init_weights()
    
    def _init_weights(self):
        """初始化模型权重"""
        for name, param in self.named_parameters():
            if 'weight' in name:
                if 'gru' in name:
                    nn.init.orthogonal_(param)
                else:
                    nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0)
    
    def forward(self, x):
        """
        前向传播
        
        参数:
        - x: 输入序列，形状为 [batch_size, seq_len, input_size]
        
        返回:
        - 预测值，形状为 [batch_size]
        """
        batch_size = x.size(0)
        
        # GRU前向传播
        gru_out, _ = self.gru(x)  # [batch_size, seq_len, hidden_size * num_directions]
        
        # 注意力机制
        attn_weights = F.softmax(self.attention(gru_out), dim=1)  # [batch_size, seq_len, 1]
        context = torch.sum(attn_weights * gru_out, dim=1)  # [batch_size, hidden_size * num_directions]
        
        # 输出层
        out = self.fc(context)  # [batch_size, 1]
        
        return out.squeeze(-1)  # [batch_size]
    

class SimpleCryptoGRU(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1, dropout=0.2):
        """
        简化的GRU模型
        
        参数:
        - input_size: 输入特征维度
        - hidden_size: GRU隐藏层大小
        - num_layers: GRU层数
        - dropout: Dropout比例
        """
        super(SimpleCryptoGRU, self).__init__()
        
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
            nn.sigmoid()
        )
    
    def forward(self, x):
        # GRU前向传播
        gru_out, _ = self.gru(x)  # [batch_size, seq_len, hidden_size]
        
        # 取最后一个时间步的输出
        last_output = gru_out[:, -1, :]  # [batch_size, hidden_size]
        
        # 输出层
        out = self.fc(last_output)  # [batch_size, 1]
        
        return out.squeeze(-1)  # [batch_size]


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

    
class ImprovedCryptoTransformer(nn.Module):
    def __init__(self, feat_dim, seq_len=96, d_model=128, nhead=8, nlayers=4, n_coins=350):
        super().__init__()
        self.seq_len = seq_len
        self.n_coins = n_coins
        self.feat_dim = feat_dim
        
        # 特征投影层
        self.feature_proj = nn.Linear(feat_dim, d_model)
        
        # 位置编码
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len)
        
        # Transformer编码器
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead,
            dim_feedforward=d_model*4,
            dropout=0.1, 
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=nlayers)
        
        # 输出头
        self.output_head = nn.Sequential(
            nn.Linear(d_model, d_model//2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(d_model//2, 1)
        )
        
    def forward(self, x):  # x: [B, N, T, F]
        B, N, T, F = x.shape
        
        # 重塑为 [B*N, T, F] 以并行处理所有币种
        x = x.reshape(B * N, T, F)
        
        # 特征投影 [B*N, T, F] -> [B*N, T, d_model]
        x = self.feature_proj(x)
        
        # 添加位置编码
        x = self.pos_encoder(x)
        
        # Transformer编码 [B*N, T, d_model] -> [B*N, T, d_model]
        x = self.transformer_encoder(x)
        
        # 取最后一个时间步的输出 [B*N, T, d_model] -> [B*N, d_model]
        x = x[:, -1, :]
        
        # 重塑回 [B, N, d_model]
        x = x.reshape(B, N, -1)
        
        # 预测头 [B, N, d_model] -> [B, N, 1] -> [B, N]
        score = self.output_head(x).squeeze(-1)
        
        return torch.sigmoid(score)

In [20]:
def train_model(model, train_loader, val_loader, num_epochs=10, learning_rate=0.001, patience=10):
    """
    训练GRU模型
    
    参数:
    - model: GRU模型实例
    - train_loader: 训练数据加载器
    - val_loader: 验证数据加载器
    - num_epochs: 训练轮数
    - learning_rate: 学习率
    - patience: 早停耐心值
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    # 定义损失函数和优化器
    criterion = nn.MSELoss()
#     optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4   # 可以尝试 1e-5 到 1e-3 范围
    )
    
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    #     optimizer, mode='min', factor=0.5, patience=5, verbose=True
    # )

    # scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    # scheduler.step()

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)

    # 记录训练历史
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(num_epochs):
        # 训练阶段
        model.train()
        train_loss = 0
        for (data, target, info) in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs} - Training"):
            data, target = data.to(device), target.to(device)
            
            # 前向传播
            output = model(data)
            loss = criterion(output, target)
            
            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            total_norm = 0
    
            
            # 梯度裁剪，防止梯度爆炸
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
        
        # 验证阶段
        model.eval()
        val_loss = 0
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for (data, target, info) in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                
                val_loss += criterion(output, target).item()
                all_preds.extend(output.cpu().numpy())
                all_targets.extend(target.cpu().numpy())
        
        # 计算平均损失
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        
        # 计算评估指标
        mse = mean_squared_error(all_targets, all_preds)
        mae = mean_absolute_error(all_targets, all_preds)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
        # 学习率调整
        scheduler.step(val_loss)
        
        # 早停检查
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # 保存最佳模型
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            patience_counter += 1
        
        # 打印训练进度
        if True:
            print(f'Epoch [{epoch+1}/{num_epochs}], '
                  f'Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}, '
                  f'MSE: {mse:.6f}, MAE: {mae:.6f}')
        
        import gc
        del data, target, output, loss
        gc.collect()
        torch.cuda.empty_cache()

        # 早停
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break
    
    # 加载最佳模型
    model.load_state_dict(torch.load('best_model.pth'))
    
    return model, train_losses, val_losses

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [25]:
full_dataset = AlphaNetUniversalDataset(df_final, features)

train_len = int(0.8 * len(full_dataset))
val_len = len(full_dataset) - train_len

train_ds, val_ds = random_split(
        full_dataset,
        [train_len, val_len],
        generator=torch.Generator().manual_seed(42)  # 保证可复现
)

train_loader = DataLoader(
    train_ds,
    batch_size=100,
    shuffle=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=100,
    shuffle=False,
)


In [23]:
# model = CryptoGRU(input_size=len(feat_cols), hidden_size=128, num_layers=2, dropout=0.3)
model = SimpleCryptoGRU(input_size=len(features))

In [26]:
trained_model, train_losses, val_losses = train_model(
    model, train_loader, val_loader, num_epochs=10, learning_rate=0.001
)

Epoch 1/10 - Training: 100%|██████████████████████████████████████████████████████| 7770/7770 [01:09<00:00, 111.35it/s]


ValueError: Input contains NaN.

In [33]:
df_final.shape

(973094, 37)

timestamp                0
open                     0
high                     0
low                      0
close                    0
volume                   0
amount                   0
count                    0
buy_volume               0
buy_amount               0
vwap                     0
symbol                   0
y_raw                    0
momentum            449510
vma                      0
obv_chg             441616
bsi                 527247
vwap_dev            420323
gk_volatility            0
avg_trade_size           0
amt_ratio                0
macd_signal         426071
bb_position         436547
williams_r          858454
stoch_k                 45
atr                      0
mfi                      0
cmf                 393748
cci                 437188
upper_shadow         11284
lower_shadow         10652
body_ratio             139
price_accel         440043
hl_spread                0
close_position         994
price_efficiency         0
intrabar_vol             0
d